# Historical market data · the first pipeline

Start here for the Market Data ML Pipeline workshop. You'll follow daily price bars from a small fixture through validation, feature calculation and orchestration. The fixture has 30 synthetic observations for one ticker, so everyone can run the same lesson.

From the repository root, run `uv sync --locked --group notebook`. Open this file in VS Code with the Jupyter extension, select the project's `.venv` kernel, then run the cells from the top. Keep the linked files open in a second editor pane when you want to follow a function.

The main exercises use the fixture, local Parquet snapshots, and a local Prefect run. Section 5 writes to a local SQLite file. Sections 1–6 form the main walkthrough. Section 7 is a short performance exercise for the transition to later sessions.

## 1. Read raw observations

**Repository source** · [`config.py`](../src/market_data/config.py) supplies the ticker and dates. [`ingest.py` → `read_fixture`](../src/market_data/ingest.py) reads the [packaged CSV](../src/market_data/data/market_sample.csv).

OHLCV stands for **open, high, low, close, volume**. One row describes one trading session. The start date is included and the end date is excluded. Look at the original column names before the pipeline renames them.

In [ ]:
from market_data.config import FIXTURE_END, FIXTURE_START, FIXTURE_SYMBOL
from market_data.ingest import read_fixture

raw = read_fixture(FIXTURE_START, FIXTURE_END)
print(f"{FIXTURE_SYMBOL}: {raw.height} rows from {FIXTURE_START} to before {FIXTURE_END}")
raw.head()

## 2. Bronze · save what arrived

**Repository source** · [`ingest.py` → `save_snapshot`](../src/market_data/ingest.py) is the function the Prefect task [`save_bronze`](../src/market_data/flows/historical_ingestion.py) calls.

Bronze keeps the original fields and adds the ticker, source and ingestion time. Parquet is a file format that preserves column types. The cell saves a file under `data/bronze/` in the project and prints its full path. It reads the file back so you can inspect the first few rows below. Every run keeps a separate snapshot.

In [ ]:
from pathlib import Path

import polars as pl

from market_data.ingest import save_snapshot

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "src" / "market_data").is_dir():
    raise RuntimeError("Open the market-data-ml-pipeline folder in VS Code and rerun.")

bronze_directory = project_root / "data" / "bronze"
snapshot_path = save_snapshot(raw, FIXTURE_SYMBOL, "fixture", bronze_directory)
print(f"Saved Bronze snapshot to {snapshot_path}")
bronze = pl.read_parquet(snapshot_path)

bronze.head()

## 3. Silver · validate and standardize

**Repository source** · [`validate.py` → `validate_prices`](../src/market_data/validate.py) prepares the rows that the flow stores in the Silver table.

Silver standardizes names and types, checks the date range and price fields, and sorts the data. The pair `(symbol, trade_date)` identifies one price row. The validator rejects a batch that repeats a key.

In [ ]:
from market_data.validate import validate_prices

silver = validate_prices(bronze, FIXTURE_SYMBOL, FIXTURE_START, FIXTURE_END)
print(silver.schema)
silver.head()

**Try one quality check** · Add a duplicate in memory and send it through the same validator in [`validate.py`](../src/market_data/validate.py). The expected error helps identify what to fix in the input.

In [ ]:
duplicate = pl.concat([raw, raw.head(1)])

try:
    validate_prices(duplicate, FIXTURE_SYMBOL, FIXTURE_START, FIXTURE_END)
except ValueError as error:
    print(error)

## 4. Gold · calculate features

**Repository source** · [`transform.py` → `build_features`](../src/market_data/transform.py) calculates each feature using the current session and earlier sessions for the same ticker.

Daily return compares today's close with the previous close. A moving average smooths the closing series over the previous 5 or 20 observations, including the current one. Five-return volatility measures the spread of recent daily returns. Early rows are null while the calculation waits for enough observations. These features can be reused in the ML session once you choose a prediction time.

In [ ]:
from market_data.transform import build_features

gold = build_features(silver)
gold.head(6)

**Explore the result** · Join the close from Silver to the features from [`transform.py`](../src/market_data/transform.py) using their shared key. This plotting code lives in the notebook. Look for the points where each moving average starts and compare them with the first visible daily return.

In [ ]:
import matplotlib.pyplot as plt

exploration = silver.join(gold, on=["symbol", "trade_date"]).sort("trade_date")
dates = exploration["trade_date"].to_list()

figure, (prices_ax, returns_ax) = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
prices_ax.plot(dates, exploration["close"].to_numpy(), label="Close")
prices_ax.plot(dates, exploration["sma_5"].to_numpy(), label="5-session mean")
prices_ax.plot(dates, exploration["sma_20"].to_numpy(), label="20-session mean")
prices_ax.set(title="Synthetic closing prices", ylabel="Example price")
prices_ax.legend()

returns_ax.bar(dates, exploration["daily_return"].to_numpy(), color="tab:blue")
returns_ax.axhline(0, color="gray", linewidth=0.8)
returns_ax.set(title="Daily change from the previous close", ylabel="Return (%)")
figure.autofmt_xdate()
figure.tight_layout()
plt.show()

## 5. Store Silver and Gold in SQLite

**Repository source** · [`storage.py`](../src/market_data/storage.py) writes both tables in one transaction using the [local SQLite schema](../src/market_data/data/sqlite_schema.sql).

A transaction commits the price rows and features together. An upsert updates an existing key or inserts a new row. The cell loads the fixture twice and checks that the counts stay the same.

The database is a file under `data/market_data.sqlite3`. Open it with a SQLite viewer, or run the SQL cell below to see the saved prices and features together. The file stays on disk after the notebook closes.

In [ ]:
from market_data.storage import load_market_data, row_counts

database_path = project_root / "data" / "market_data.sqlite3"
load_market_data(database_path, silver, "fixture")
first_counts = row_counts(database_path, FIXTURE_SYMBOL)

load_market_data(database_path, silver, "fixture")
second_counts = row_counts(database_path, FIXTURE_SYMBOL)

assert first_counts == second_counts
print(f"SQLite database: {database_path}")
pl.DataFrame(second_counts, schema=["table", "rows", "distinct_keys"], orient="row")

**Read the stored data** · The SQL joins the two tables on their shared key. The question mark binds the fixture symbol as a value. The query shows the first six days, including the empty fields while the moving averages warm up.

In [ ]:
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(database_path)) as connection:
    stored_prices = pl.read_database(
        query="""
        SELECT p.trade_date, p.close, f.daily_return, f.sma_5, f.sma_20
        FROM market_prices AS p
        JOIN market_features AS f USING (symbol, trade_date)
        WHERE p.symbol = ?
        ORDER BY p.trade_date
        LIMIT 6
        """,
        connection=connection,
        execute_options={"parameters": (FIXTURE_SYMBOL,)},
    )

stored_prices

## 6. Run the steps as a Prefect flow

**Repository source** · [`flows/historical_ingestion.py` → `ingest_historical_prices`](../src/market_data/flows/historical_ingestion.py). [`cli.py` → `run_command`](../src/market_data/cli.py) calls this same flow from the terminal.

A flow is the complete run. Tasks are the steps Prefect tracks inside it. Read the task calls at the bottom of the file and compare them with the cells above. The preview uses a temporary local Prefect server by default. If you started the local server from the [README](../README.md#prefect-ui-and-local-daily-schedule), set `local_api_url` to `"http://127.0.0.1:4200/api"` to see the run in its UI.

This example skips a second Bronze write because you already explored the file in Section 2. The Cloud configuration makes the same choice. Its worker still fetches Yahoo data, then the first stored layer is Silver. The SQLite exercise above shows how Silver and Gold are committed together. For deployment, the worker gets the Supabase connection string from a [Prefect Secret block](../docs/DEPLOYMENT.md#create-the-supabase-secret-block).

In [ ]:
from prefect.settings import (
    PREFECT_API_KEY,
    PREFECT_API_URL,
    PREFECT_SERVER_ANALYTICS_ENABLED,
    PREFECT_SERVER_EPHEMERAL_ENABLED,
    temporary_settings,
)

from market_data.flows.historical_ingestion import ingest_historical_prices

local_api_url = None
with temporary_settings(
    updates={
        PREFECT_API_URL: local_api_url,
        PREFECT_API_KEY: None,
        PREFECT_SERVER_ANALYTICS_ENABLED: False,
        PREFECT_SERVER_EPHEMERAL_ENABLED: True,
    }
):
    flow_prices, flow_features, load_counts = ingest_historical_prices(
        source="fixture", skip_db=True, write_bronze=False
    )

assert flow_prices.equals(silver)
assert flow_features.equals(gold)
print(f"The flow reproduced {flow_prices.height} Silver rows and {flow_features.height} Gold rows.")

**Try a missed batch** · The scheduled deployment uses `incremental: true`. It checks the latest committed trade date for each ticker, rereads seven calendar days of overlap, including that date, and fetches forward to the current UTC date, which is excluded from the window. The fixed fixture lets you set the cutoff yourself.

In the [missed-run exercise](../README.md#practice-catch-up-after-missed-runs), save the first ten fixture rows and pretend the worker was offline for two weeks. Catch-up recovers the ten missing rows, and a rerun leaves the row count at twenty. Look for **latest stored date** and **Fetching** in the logs. A *manual backfill* takes dates you choose, which is useful for an older gap inside saved history.

For a simulated timeout retry, a validation failure, and a short explanation of `NO_CACHE`, see the [instructor guide](../docs/INSTRUCTOR_DEMO.md#prefect-terms-for-the-workshop).

## 7. Compare the same rolling mean in pandas and Polars

**Repository source** · The 20-session mean comes from [`transform.py` → `build_features`](../src/market_data/transform.py). The notebook isolates that one calculation on a single numeric series. It uses the same values, window and warm-up rule in both libraries.

First check that the results agree. Then time only the calculation, with the two Series already in memory. The larger inputs repeat the fixture values to give the operation more work. They are synthetic benchmark inputs.

In [ ]:
from statistics import median
from timeit import repeat

import numpy as np
import pandas as pd

window = 20
close_values = silver["close"].to_numpy()
pandas_close = pd.Series(close_values)
polars_close = pl.Series("close", close_values)

pandas_mean = pandas_close.rolling(window, min_periods=window).mean()
polars_mean = polars_close.rolling_mean(window_size=window, min_samples=window)
np.testing.assert_allclose(pandas_mean.to_numpy(), polars_mean.to_numpy(), equal_nan=True)
print("Both libraries agree, including the warm-up rows.")

**Time the same operation at three sizes.** Each number is the median time in milliseconds for one calculation, estimated from five trials of five calls. **Lower is faster.** Data generation, conversion and disk reads are outside the timed section. Both libraries run once at each size before timing so you can check that their outputs agree.

In [ ]:
timings = []
for copies in (1, 1_000, 10_000):
    values = np.tile(close_values, copies)
    pandas_close = pd.Series(values)
    polars_close = pl.Series("close", values)

    pandas_result = pandas_close.rolling(window, min_periods=window).mean()
    polars_result = polars_close.rolling_mean(window_size=window, min_samples=window)
    np.testing.assert_allclose(pandas_result.to_numpy(), polars_result.to_numpy(), equal_nan=True)

    pandas_times = repeat(
        lambda series=pandas_close: series.rolling(window, min_periods=window).mean(),
        number=5,
        repeat=5,
    )
    polars_times = repeat(
        lambda series=polars_close: series.rolling_mean(window_size=window, min_samples=window),
        number=5,
        repeat=5,
    )
    pandas_ms = median(pandas_times) * 1_000 / 5
    polars_ms = median(polars_times) * 1_000 / 5
    faster = "pandas" if pandas_ms < polars_ms else "Polars"
    timings.append(
        {
            "rows": len(values),
            "pandas_ms": pandas_ms,
            "polars_ms": polars_ms,
            "faster": faster,
        }
    )

print(f"pandas {pd.__version__} | Polars {pl.__version__}")
pl.DataFrame(timings)

Each row compares the libraries for one operation on your machine. It is possible for Polars to win at 30 rows and pandas to win at 300,000. At 30 rows, setup overhead and timing noise can be a large share of the measurement. At larger sizes, the implementation of this rolling calculation matters. Both libraries implement it in native code, and an eager calculation on one column does little to exercise Polars' lazy query optimization or parallel work across expressions.

A benchmark of a complete workload would include whatever your job actually does, such as reading files, filtering, joining and aggregating. Try measuring that workload when deciding which library to use for it. Later, with PySpark, you can measure the added time for starting workers, partitioning data and moving it across machines. [Polars guide for pandas users](https://docs.pola.rs/user-guide/migration/pandas/)